# Module 1 — Agent Fundamentals

**Course:** Build Your First AI Agent from Scratch  
**Community:** [Autonomous MSP](https://skool.com/autonomous-msp-2162)

This notebook has **no API calls**. It illustrates the core concepts with plain Python so you can run every cell without an API key.

**What you will learn:**
- Why scripts break and agents don't
- The four pillars of an agent (Reasoning Engine, Memory, Tools, Context)
- The three reasoning patterns: ReAct, Planning, Reflection
- What the Perception-Action loop looks like in code

## Lesson 1 — Scripts vs Agents

A script is a decision tree you drew in advance. Every possible branch must exist **before** the problem happens.

An agent is a reasoning loop. It looks at what it finds, decides what to do next, and keeps going — even if it has never seen this exact situation before.

Run the two cells below to see the difference.

In [ ]:
# ── SCRIPT approach ────────────────────────────────────────────
# A script only handles cases you predicted in advance.

def script_handle_ospf(neighbor_state: str):
    """
    Classic script: a static decision tree.
    If the state is not in our if/elif chain → silent exit.
    """
    if neighbor_state == "FULL":
        print("OK — neighbor is healthy.")
    elif neighbor_state == "DOWN":
        print("ALERT — neighbor is down. Paging NOC.")
    else:
        # INIT, ATTEMPT, EXSTART — all fall here
        print("Unexpected state. Exiting cleanly.")
        # Ticket stays open. You wake up.


print("Script response to each OSPF state:")
for state in ["FULL", "DOWN", "INIT", "EXSTART", "ATTEMPT"]:
    print(f"  State={state}: ", end="")
    script_handle_ospf(state)

In [ ]:
# ── AGENT mindset ──────────────────────────────────────────────
# An agent reasons about what it finds, then decides the next step.
# It does not need a pre-written branch for every state.

def agent_handle_ospf(neighbor_state: str, available_tools: list):
    """
    Simulates how an agent thinks:
    - Look at the state
    - Form a hypothesis
    - Pick the right tool to investigate further
    """
    print(f"  Neighbor state: {neighbor_state}")

    # The agent reasons based on protocol knowledge
    if neighbor_state == "FULL":
        print("  Thought: Adjacency is healthy. No action needed.")

    elif neighbor_state == "INIT":
        print("  Thought: INIT means one-way Hello exchange.")
        print("           Possible causes: area mismatch, MTU mismatch, auth mismatch.")
        print("  Action: I will check both sides of the link.")
        if "check_interface_config" in available_tools:
            print("  Tool call → check_interface_config(both sides)")

    elif neighbor_state == "EXSTART":
        print("  Thought: EXSTART means DBD exchange is failing.")
        print("           Most common cause: MTU mismatch.")
        print("  Action: I will check MTU on both interfaces.")
        if "check_mtu" in available_tools:
            print("  Tool call → check_mtu(both sides)")

    elif neighbor_state == "ATTEMPT":
        print("  Thought: ATTEMPT is common on NBMA networks.")
        print("           Could be unicast Hello issue. I need to check the network type.")
        if "check_ospf_network_type" in available_tools:
            print("  Tool call → check_ospf_network_type()")

    else:
        print(f"  Thought: Unfamiliar state '{neighbor_state}'.")
        print("           I will gather more data before concluding anything.")


my_tools = ["check_interface_config", "check_mtu", "check_ospf_network_type", "ping"]

print("Agent response to each OSPF state:\n")
for state in ["FULL", "INIT", "EXSTART", "ATTEMPT", "EXCHANGE"]:
    agent_handle_ospf(state, my_tools)
    print()

## Lesson 2 — The Four Pillars of an Agent

Every agent needs four things to work:

| Pillar | What it does | Example |
|--------|-------------|--------|
| **Reasoning Engine** | Decides what to do next | Claude, GPT-4 |
| **Memory** | Keeps context across steps | Past tickets, topology docs |
| **Tools** | Lets the agent interact with the world | SSH, API calls, DB queries |
| **Context** | Everything the agent knows right now | Ticket text, device outputs |

Remove any one of these and you have something weaker than an agent.

In [ ]:
# ── Illustrating the four pillars with plain Python ────────────

# Pillar 1: Reasoning Engine (simulated — no real LLM here)
def reasoning_engine(observation: str, context: dict) -> str:
    """
    In production this is an LLM call.
    Here we simulate the output to illustrate the pattern.
    """
    if "INIT" in observation:
        return "check_both_interface_areas"
    if "area mismatch" in observation.lower():
        return "DONE"
    return "check_interface_config"


# Pillar 2: Memory (simplified — a list of past incidents)
memory = [
    {"symptom": "OSPF INIT state", "root_cause": "Area mismatch", "ticket": "INC-1955"},
    {"symptom": "BGP session drop", "root_cause": "Route-map deny", "ticket": "INC-2341"},
]

def search_memory(query: str) -> list:
    """In production this is a ChromaDB vector search."""
    return [m for m in memory if query.lower() in m["symptom"].lower()]


# Pillar 3: Tools (simple functions the agent can call)
tools = {
    "check_both_interface_areas": lambda: "R1 Gi0/1: Area 0 | R2 Gi0/0: Area 1",
    "check_interface_config": lambda: "Hello 10, Dead 40, MTU 1500",
    "ping": lambda: "Success rate 100%",
}


# Pillar 4: Context (everything available to the agent at this moment)
context = {
    "ticket": "OSPF neighbor 3.3.3.3 on R1 stuck in INIT state",
    "device": "R1",
    "relevant_memory": search_memory("OSPF INIT"),
}


# ── Putting it all together ────────────────────────────────────
print("=== AGENT RUN (simulated) ===")
print(f"Ticket: {context['ticket']}")
print(f"Memory hit: {context['relevant_memory']}\n")

observation = "R1 neighbor 3.3.3.3 is in INIT state"

for step in range(1, 5):
    action = reasoning_engine(observation, context)
    print(f"Step {step}: Reasoning engine → '{action}'")

    if action == "DONE":
        print("\nConclusion: Area mismatch confirmed. R1 Gi0/1=Area 0, R2 Gi0/0=Area 1.")
        print("Fix: Set both interfaces to the same OSPF area.")
        break

    if action in tools:
        observation = tools[action]()
        print(f"         Tool result  → '{observation}'")
    print()

## Lesson 3 — The Three Reasoning Patterns

**ReAct** (Reason + Act) — for troubleshooting. Evidence in → decision out → next action → repeat.  
**Planning** — for multi-step deployment. Decompose the goal first, then execute.  
**Reflection** — for validation. Generate output, then critique it before delivering.  

You will use ReAct 80% of the time. The next cell shows what each pattern looks like.

In [ ]:
# ── Pattern 1: ReAct (Troubleshooting) ────────────────────────
print("=== PATTERN 1: ReAct (Troubleshooting) ===")
react_steps = [
    ("Thought", "OSPF neighbor in INIT. Could be area mismatch, MTU, or auth."),
    ("Action",  "show_ospf_interface(R1, Gi0/1)"),
    ("Observation", "Area 0, Hello 10, Dead 40"),
    ("Thought", "R1 is Area 0. Check R2's side."),
    ("Action",  "show_ospf_interface(R2, Gi0/0)"),
    ("Observation", "Area 1, Hello 10, Dead 40"),
    ("Conclusion", "Area mismatch. R1=Area 0, R2=Area 1. Fix: align areas."),
]
for label, text in react_steps:
    print(f"  {label:12s}: {text}")

print()

# ── Pattern 2: Planning (Deployment) ──────────────────────────
print("=== PATTERN 2: Planning (Multi-site VLAN deployment) ===")
plan = [
    "1. Retrieve current VLAN config from all 12 branch switches.",
    "2. Identify conflicts with existing VLAN IDs.",
    "3. Generate config templates for each device.",
    "4. Run dry-run mode, collect diffs.",
    "5. Present diffs to engineer for approval.",
    "6. On approval: push configs in sequence with rollback ready.",
]
for step in plan:
    print(f"  {step}")

print()

# ── Pattern 3: Reflection (Validation) ────────────────────────
print("=== PATTERN 3: Reflection (Config Validation) ===")
proposed_config = "interface Gi0/1\n  ip ospf 1 area 1"
checks = [
    ("Change freeze active?",       "No → proceed"),
    ("Conflicts with existing policy?", "No → proceed"),
    ("Rollback path documented?",   "Yes → interface was Area 0 before"),
    ("Matches org OSPF standard?",  "Yes → process 1, standard areas"),
]
print(f"  Proposed config: {repr(proposed_config)}")
print("  Reflection checks:")
for check, result in checks:
    print(f"    - {check:40s} {result}")
print("  Verdict: Config is safe to present to engineer for final approval.")

## What's Next

You now have the mental model. In **Module 2** you will wire a real LLM into the ReAct loop and build the first working agent — in pure Python, no frameworks, ~80 lines.

---

**Module Challenge:** Write down your single most repetitive tier-1 troubleshooting task — the one you wish you never had to do manually again. Be specific: not "network issues", but something like "BGP session drops on Meraki MX after firmware update". Post it in **#agent-builds**.